# LifeMate — chạy agent sau ASR

1. Chạy `predict_asr.py` cùng thư mục để sinh đủ prediction hiện tại/history.
2. Nhóm trưởng điền `load_toolenv()` trong `src/lifemate/agent.py`.
3. Sửa cấu hình bên dưới và chạy toàn bộ split.

Code agent nằm ở `src/lifemate/agent.py`; notebook chỉ gọi code đó, không giữ một
bản sao khác. ASR và agent dùng chung đường dẫn qua `lifemate.paths`.
Hướng dẫn đầy đủ: `docs/lifemate/README.md`. Chưa xác minh suy luận GPU end-to-end.


In [ ]:
import os
from pathlib import Path
import sys

# PROJECT_ROOT là repo source, khác DATA_ROOT là dữ liệu BTC.
# Nếu notebook nằm ngoài repo, đặt ZAIC_PROJECT_ROOT tới repo trước khi chạy cell.
# Local hiện tại: C:/zalo_ai/ZAIC-76Vibecoders; Kaggle: thư mục repo đã đưa vào phiên.
start = Path(os.environ.get("ZAIC_PROJECT_ROOT", str(Path.cwd()))).resolve()
PROJECT_ROOT = next((root for root in [start, *start.parents]
                     if (root / "pyproject.toml").is_file() and (root / "src/lifemate").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Đặt ZAIC_PROJECT_ROOT tới thư mục ZAIC-76Vibecoders")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from lifemate.agent import ModelLoadConfig, run_full_pipeline
from lifemate.paths import default_data_root, asr_prediction_file, agent_output_dir


## Cấu hình

SPLIT chọn `train` hoặc `public_test`; mặc định chạy toàn bộ split.
DATA_ROOT cần trỏ tới thư mục chứa các split. MODEL_SOURCE là checkpoint/tokenizer
LLM đã có trong local/cache. ASR_FILE là file text ASR đã có; cell chạy agent
chỉ đọc file này. Nếu file đến từ máy/notebook khác, sửa ASR_FILE tới nơi đã chuyển.


In [ ]:
# DATA_ROOT chứa train/, public_test/; KHÔNG thêm /train vào cuối.
# Local: C:/zalo_ai/train+public_test
# Kaggle: /kaggle/input/datasets/khangl123/zalo-ai/train+public_test
# Muốn đổi: gán Path("...") trực tiếp hoặc đặt LIFEMATE_DATA_ROOT trước cell này.
DATA_ROOT = default_data_root()
SPLIT = "train"

# ID này chỉ chạy offline khi model + tokenizer đã có trong cache.
# Hoặc thay bằng thư mục checkpoint thật trên local / model đã Add Input trên Kaggle.
MODEL_SOURCE = "Qwen/Qwen3-4B-Instruct-2507"

# File text ASR cùng SPLIT, đã có đủ current/history. Đổi path nếu chuyển từ nơi khác.
# Local: <repo>/output/lifemate/asr/<split>/asr_predictions.jsonl
# Kaggle: /kaggle/working/lifemate/asr/<split>/asr_predictions.jsonl
# File từ Kaggle notebook khác: Add Input rồi gán ASR_FILE tới file dưới /kaggle/input.
ASR_FILE = asr_prediction_file(SPLIT, PROJECT_ROOT)
# Local: <repo>/output/lifemate/agent; Kaggle: /kaggle/working/lifemate/agent.
# Runner tự thêm <run_id>/; /kaggle/input chỉ đọc, không dùng để ghi kết quả.
AGENT_OUTPUT = agent_output_dir(PROJECT_ROOT)
print("Dữ liệu:", DATA_ROOT)
print("Input ASR:", ASR_FILE)
print("Model agent:", MODEL_SOURCE)
print("Output agent:", AGENT_OUTPUT)


## Chạy

File ASR thiếu/sai ID/history sẽ bị chặn trước khi nạp Qwen. `load_toolenv()` vẫn
là chỗ nhóm trưởng điền; chưa điền thì chương trình báo NotImplementedError.
Không có fallback sang transcript chuẩn hay label. Trace được lưu riêng.


In [ ]:
report = run_full_pipeline(
    data_root=DATA_ROOT,
    split=SPLIT,
    asr_predictions_path=ASR_FILE,
    output_dir=AGENT_OUTPUT,
    model_config=ModelLoadConfig(model_source=MODEL_SOURCE, local_files_only=True),
)
print("JSONL:", report["submission_path"])
print("Trace:", report["trace_dir"])
